In [ ]:
import numpy as np 

import matplotlib.pyplot as plt

from delta_sigma_simulator.modulator import DeltaSigmaModulator
from delta_sigma_simulator.filter import FilterIntegrator, FilterFirstOrder, FilterSecondOrder
from delta_sigma_simulator.quantizer import QuantizerDelayHysteresis
from delta_sigma_simulator.wave import BinaryWave

In [ ]:
beta_sim = np.logspace(0, 3, 20) 
beta = np.logspace(0, 3, 1000)

f0 = 1

In [ ]:
f0_sim = np.zeros_like(beta_sim)
f2_sim = np.zeros_like(beta_sim)

V1_sim = np.zeros_like(beta_sim)
V3_sim = np.zeros_like(beta_sim)

for i, beta_i in enumerate(beta_sim):
    loop_filter = FilterIntegrator(2 * np.pi * f0 / beta_i)
    quantizer = QuantizerDelayHysteresis(0, np.pi / 2 / beta_i)
    quantizer.t_step = 0.1
    asdm = DeltaSigmaModulator(loop_filter, quantizer)

    U = np.linspace(-0.01, 0.01, 51)
    f_sim = np.zeros_like(U)
    V_sim = np.zeros_like(U)

    for j, U_j in enumerate(U):
        v = asdm.simulate([BinaryWave(E=U_j)], n=10)

        assert np.isclose(v.e[-1] - v.e[-3], v.e[-3] - v.e[-5]), "Simulation failed to converge."
            
        f_sim[j] = 1 / (v.e[-1] - v.e[-3])
        V_sim[j] = -v.E * (-1 + 2 * (v.e[-2] - v.e[-3]) / (v.e[-1] - v.e[-3]))

    res = np.polynomial.polynomial.polyfit(V_sim, f_sim, 7)

    f0_sim[i] = f_sim[len(f_sim) // 2]
    f2_sim[i] = res[2]
    
    res = np.polynomial.polynomial.polyfit(U, V_sim, 7)

    V1_sim[i] = res[1]
    V3_sim[i] = res[3]

In [ ]:
f2 = -1 * np.ones_like(beta)

V1 = 1 * np.ones_like(beta)
V3 = 0 * np.ones_like(beta)

In [ ]:
plt.semilogx(beta_sim, f0_sim, 'ko')
plt.semilogx(beta, f0 * np.ones_like(beta), 'k-')

plt.semilogx(beta_sim, f2_sim / f0_sim, 'ro')
plt.semilogx(beta, f2 / f0, 'r-')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
plt.semilogx(beta_sim, V1_sim, 'ko')
plt.semilogx(beta, V1, 'k-')

plt.semilogx(beta_sim, V3_sim, 'ro')
plt.semilogx(beta, V3, 'r-')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
np.savetxt("asdm-loop-filter-integrator-simulation.csv", np.column_stack((beta_sim, f0_sim, f2_sim, V1_sim, V3_sim)), delimiter=",")
np.savetxt("asdm-loop-filter-integrator-expression.csv", np.column_stack((beta, f0 * np.ones_like(beta), f2, V1, V3)), delimiter=",")

In [ ]:
f0_sim = np.zeros_like(beta_sim)
f2_sim = np.zeros_like(beta_sim)

V1_sim = np.zeros_like(beta_sim)
V3_sim = np.zeros_like(beta_sim)

for i, beta_i in enumerate(beta_sim):
    loop_filter = FilterFirstOrder(f0 / beta_i)
    quantizer = QuantizerDelayHysteresis(0, np.tanh(np.pi / 2 / beta_i))
    quantizer.t_step = 0.1
    asdm = DeltaSigmaModulator(loop_filter, quantizer)

    U = np.linspace(-0.01, 0.01, 51)
    f_sim = np.zeros_like(U)
    V_sim = np.zeros_like(U)

    for j, U_j in enumerate(U):
        v = asdm.simulate([BinaryWave(E=U_j)], n=10)

        assert np.isclose(v.e[-1] - v.e[-3], v.e[-3] - v.e[-5]), "Simulation failed to converge."
            
        f_sim[j] = 1 / (v.e[-1] - v.e[-3])
        V_sim[j] = -v.E * (-1 + 2 * (v.e[-2] - v.e[-3]) / (v.e[-1] - v.e[-3]))

    res = np.polynomial.polynomial.polyfit(V_sim, f_sim, 7)

    f0_sim[i] = f_sim[len(f_sim) // 2]
    f2_sim[i] = res[2]
    
    res = np.polynomial.polynomial.polyfit(U, V_sim, 7)

    V1_sim[i] = res[1]
    V3_sim[i] = res[3]

In [ ]:
f2 = -np.pi / (2 * beta) / np.tanh(np.pi / 2 / beta)

V1 = np.sinh(np.pi / beta) / (np.pi / beta)
V3 = np.pi ** 2 / 6 / beta ** 2 * (3 / 2 / np.tanh(np.pi / 2 / beta) ** 2 - 3 * beta / np.pi / np.tanh(np.pi / 2 / beta) + 1 / 2) * V1 ** 3

alpha = np.abs(1 + 1j * f0 / (f0 / beta))

V1_2 = 1 / (1 - np.pi ** 2 / alpha / beta)
V3_2 = np.pi ** 2 / 6 / alpha / beta / (1 - np.pi ** 2 / 6 / alpha / beta) * V1_2 ** 3

In [ ]:
plt.semilogx(beta_sim, f0_sim, 'ko')
plt.semilogx(beta, f0 * np.ones_like(beta), 'k-')

plt.semilogx(beta_sim, f2_sim / f0_sim, 'ro')
plt.semilogx(beta, f2 / f0, 'r-')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
plt.loglog(beta_sim, np.abs(V1_sim), 'ko')
plt.loglog(beta, np.abs(V1), 'k-')
plt.loglog(beta, np.abs(V1_2), 'k--')

plt.loglog(beta_sim, np.abs(V3_sim), 'ro')
plt.loglog(beta, np.abs(V3), 'r-')
plt.loglog(beta, np.abs(V3_2), 'r--')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
np.savetxt("asdm-loop-filter-low-pass-simulation.csv", np.column_stack((beta_sim, f0_sim, f2_sim, V1_sim, V3_sim)), delimiter=",")
np.savetxt("asdm-loop-filter-low-pass-expression.csv", np.column_stack((beta, f0 * np.ones_like(beta), f2, V1, V3, V1_2, V3_2)), delimiter=",")

In [ ]:
sweep_sim = np.logspace(np.log10(2 / 4), np.log10(1000 / 4), 20)

beta_sim = np.zeros_like(sweep_sim)

f0_sim = np.zeros_like(sweep_sim)
f2_sim = np.zeros_like(sweep_sim)

V1_sim = np.zeros_like(sweep_sim)
V3_sim = np.zeros_like(sweep_sim)

for i, sweep_i in enumerate(sweep_sim):
    loop_filter = FilterSecondOrder(f0 / sweep_i, f0 / sweep_i / 2, f0 / sweep_i / 4, 1)

    fg = f0 / sweep_i - f0 / sweep_i / 2 - f0 / sweep_i / 4

    beta_sim[i] = f0 / fg

    quantizer = QuantizerDelayHysteresis(0, np.pi / 2 * np.abs(loop_filter.frequency_response(2 * np.pi * f0)))
    quantizer.t_step = 0.1
    asdm = DeltaSigmaModulator(loop_filter, quantizer)

    U = np.linspace(-0.1, 0.1, 51)
    f_sim = np.zeros_like(U)
    V_sim = np.zeros_like(U)

    for j, U_j in enumerate(U):
        v = asdm.simulate([BinaryWave(E=U_j)], n=20)

        assert np.isclose(v.e[-1] - v.e[-3], v.e[-3] - v.e[-5]), "Simulation failed to converge."
            
        f_sim[j] = 1 / (v.e[-1] - v.e[-3])
        V_sim[j] = -v.E * (-1 + 2 * (v.e[-2] - v.e[-3]) / (v.e[-1] - v.e[-3]))

    res = np.polynomial.polynomial.polyfit(V_sim, f_sim, 7)

    f0_sim[i] = f_sim[len(f_sim) // 2]
    f2_sim[i] = res[2]
    
    res = np.polynomial.polynomial.polyfit(U, V_sim, 7)

    V1_sim[i] = res[1]
    V3_sim[i] = res[3]

In [ ]:
np.real(loop_filter.frequency_response(0))

In [ ]:
sweep = np.logspace(np.log10(2 / 4), np.log10(1000 / 4), 1000)

f2 = -f0 * np.ones_like(sweep)

alpha = np.zeros_like(sweep)
beta  = np.zeros_like(sweep)

for i, sweep_i in enumerate(sweep):
    loop_filter = FilterSecondOrder(f0 / sweep_i, f0 / sweep_i / 2, f0 / sweep_i / 4, 1)

    fg = f0 / sweep_i - f0 / sweep_i / 2 - f0 / sweep_i / 4

    alpha[i] = np.abs(loop_filter.frequency_response(0) / loop_filter.frequency_response(2 * np.pi * f0))
    beta[i]  = f0 / fg

V1 = 1 / (1 + np.pi ** 2 / alpha / beta)
V3 = -np.pi ** 2 / 6 / alpha / beta / (1 + np.pi ** 2 / 6 / alpha / beta) * V1 ** 3

In [ ]:
plt.semilogx(beta_sim, f0_sim, 'ko')
plt.semilogx(beta, f0 * np.ones_like(beta), 'k-')

plt.semilogx(beta_sim, f2_sim, 'ro')
plt.semilogx(beta, f2, 'r-')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
plt.loglog(beta_sim, np.abs(V1_sim), 'ko')
plt.loglog(beta, np.abs(V1), 'k-')

plt.loglog(beta_sim, np.abs(V3_sim), 'ro')
plt.loglog(beta, np.abs(V3), 'r-')

plt.grid(which="both", ls="--", alpha=0.5)
plt.show()

In [ ]:
np.savetxt("asdm-loop-filter-general-simulation.csv", np.column_stack((beta_sim, f0_sim, f2_sim, V1_sim, V3_sim)), delimiter=",")
np.savetxt("asdm-loop-filter-general-expression.csv", np.column_stack((beta, beta * fg, f2, V1, V3)), delimiter=",")